# 🧠 Brain Tumor MRI Classification
## EfficientNet-B3 + CBAM Attention — Research Pipeline

| Item | Detail |
|---|---|
| **Backbone** | EfficientNet-B3 (ImageNet pretrained) |
| **Attention** | CBAM (Channel + Spatial) |
| **Training** | Two-phase: freeze → full fine-tune |
| **Loss** | CrossEntropy + Label Smoothing (ε=0.1) |
| **Optimizer** | AdamW + Cosine Annealing |
| **Target** | Beat CEGA-EfficientNetB3 → 99.39% (Msoud 4-class, 2025) |

**Fix vs original script:** proper val split (15% of Training), test set is genuinely held out.


## Cell 1 — Install Dependencies

In [ ]:
!pip install torch torchvision scikit-learn seaborn matplotlib --quiet


## Cell 2 — Imports

In [ ]:
import os
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from torchvision.models import EfficientNet_B3_Weights

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc,
    matthews_corrcoef, cohen_kappa_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
print("✅ All imports successful")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")


## Cell 3 — Configuration (edit here)

In [ ]:
# ─── Edit these paths and hyperparameters ────────────────────────────────────
DATA_DIR       = "./DATASETS/Brain Tumor MRI Dataset"  # folder with Training/ and Testing/
OUTPUT_PATH    = "best_model.pth"
PLOT_DIR       = "plots"

EPOCHS_PHASE1  = 10        # backbone frozen  — head + CBAM only
EPOCHS_PHASE2  = 20        # full fine-tune
BATCH_SIZE     = 32
LR_PHASE1      = 1e-3      # higher LR — only head is training
LR_PHASE2      = 5e-5      # low LR   — full model fine-tune
WEIGHT_DECAY   = 1e-4
VAL_FRACTION   = 0.15      # 15% of Training split used as validation
LABEL_SMOOTHING= 0.1
SEED           = 42
# ─────────────────────────────────────────────────────────────────────────────

CLASSES      = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES  = len(CLASSES)
CLASS_COLORS = ["#E63946", "#457B9D", "#2A9D8F", "#E9C46A"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Device      : {DEVICE}")
print(f"Data dir    : {DATA_DIR}")
print(f"Phase 1 eps : {EPOCHS_PHASE1}  (lr={LR_PHASE1})")
print(f"Phase 2 eps : {EPOCHS_PHASE2}  (lr={LR_PHASE2})")
print(f"Output      : {OUTPUT_PATH}")


## Cell 4 — CBAM Attention Module

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention (Hu et al., 2018)."""
    def __init__(self, in_planes: int, ratio: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(in_planes // ratio, 8)
        self.fc = nn.Sequential(
            nn.Linear(in_planes, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_planes, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return self.sigmoid((avg + mx).view(b, c, 1, 1)) * x


class SpatialAttention(nn.Module):
    """Spatial attention over channel-pooled feature maps (Woo et al., 2018)."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg     = x.mean(dim=1, keepdim=True)
        mx, _   = x.max(dim=1,  keepdim=True)
        spatial = torch.cat([avg, mx], dim=1)
        return self.sigmoid(self.conv(spatial)) * x


class CBAM(nn.Module):
    """Convolutional Block Attention Module: channel → spatial (Woo et al., ECCV 2018)."""
    def __init__(self, in_planes: int, ratio: int = 16, kernel_size: int = 7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))

print("✅ CBAM modules defined")


## Cell 5 — Model: EfficientNet-B3 + CBAM

In [ ]:
class BrainTumorNet(nn.Module):
    """
    EfficientNet-B3 backbone with CBAM injected before the classifier.
    Head: BN → Dropout(0.4) → Linear(1536→512) → GELU → BN → Dropout(0.3) → Linear(512→4)
    """
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        base          = models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.features = base.features
        self.avgpool  = base.avgpool
        in_features   = base.classifier[1].in_features   # 1536 for B3

        self.cbam = CBAM(1536, ratio=16)

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        return self.classifier(x)

    def freeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.features.parameters():
            p.requires_grad = True


# Quick sanity check
_m = BrainTumorNet(NUM_CLASSES)
_x = torch.randn(2, 3, 224, 224)
_y = _m(_x)
assert _y.shape == (2, NUM_CLASSES), f"Unexpected output shape: {_y.shape}"
del _m, _x, _y

total_params = sum(p.numel() for p in BrainTumorNet().parameters())
print(f"✅ BrainTumorNet defined")
print(f"   Total parameters : {total_params:,}")


## Cell 6 — Loss Function (Label Smoothing)

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """
    Cross-entropy with label smoothing (ε=0.1).
    Reduces overconfidence; consistently improves generalization on small medical datasets.
    """
    def __init__(self, classes: int, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing
        self.classes   = classes

    def forward(self, pred, target):
        with torch.no_grad():
            smooth = torch.full(
                (pred.size(0), self.classes),
                self.smoothing / (self.classes - 1),
                device=pred.device,
            )
            smooth.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        log_prob = F.log_softmax(pred, dim=1)
        return -(smooth * log_prob).sum(dim=1).mean()

CRITERION = LabelSmoothingLoss(NUM_CLASSES, smoothing=LABEL_SMOOTHING)
print(f"✅ LabelSmoothingLoss defined  (ε={LABEL_SMOOTHING})")


## Cell 7 — Data Transforms & Loaders

In [ ]:
def get_transforms(phase: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if phase == "train":
        return transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.RandomCrop(280),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


def get_dataloaders(data_dir, batch_size, val_fraction=0.15):
    """
    Proper split: Training → (train 85%, val 15%), Testing → held-out test.
    Fixes the test-as-validation leakage in naive implementations.
    """
    train_dir = os.path.join(data_dir, "Training")
    test_dir  = os.path.join(data_dir, "Testing")

    full_train = datasets.ImageFolder(train_dir, transform=get_transforms("train"))
    if sorted(full_train.classes) != sorted(CLASSES):
        raise ValueError(f"Class mismatch: found {full_train.classes}")

    n_total = len(full_train)
    n_val   = int(n_total * val_fraction)
    n_train = n_total - n_val
    train_ds, val_ds_idx = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )
    # Give val subset its own (non-augmented) transform
    val_ds_full = datasets.ImageFolder(train_dir, transform=get_transforms("val"))
    val_ds_idx.dataset = val_ds_full

    test_ds = datasets.ImageFolder(test_dir, transform=get_transforms("val"))

    train_loader = DataLoader(train_ds,  batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds_idx, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader, full_train.classes, full_train, test_ds


TRAIN_LOADER, VAL_LOADER, TEST_LOADER, CLASSES_FOUND, FULL_TRAIN_DS, TEST_DS_RAW = \
    get_dataloaders(DATA_DIR, BATCH_SIZE, VAL_FRACTION)

print(f"✅ Dataloaders ready")
print(f"   Classes  : {CLASSES_FOUND}")
print(f"   Train    : {len(TRAIN_LOADER.dataset):,} images")
print(f"   Val      : {len(VAL_LOADER.dataset):,} images")
print(f"   Test     : {len(TEST_LOADER.dataset):,} images")


## Cell 8 — Dataset Analysis Plots

In [ ]:
train_counts = Counter(FULL_TRAIN_DS.targets)
test_counts  = Counter(TEST_DS_RAW.targets)
train_vals   = [train_counts[i] for i in range(NUM_CLASSES)]
test_vals    = [test_counts[i]  for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Brain Tumor MRI — Dataset Analysis", fontsize=15, fontweight="bold")

# Bar chart
x, w = np.arange(NUM_CLASSES), 0.35
axes[0].bar(x - w/2, train_vals, w, label="Train", color=CLASS_COLORS, alpha=0.85, edgecolor="black")
axes[0].bar(x + w/2, test_vals,  w, label="Test",  color=CLASS_COLORS, alpha=0.5,  edgecolor="black", hatch="//")
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASSES, rotation=20)
axes[0].set_ylabel("Image Count"); axes[0].set_title("Class Distribution (Train vs Test)")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
for xi, (tv, tsv) in enumerate(zip(train_vals, test_vals)):
    axes[0].text(xi - w/2, tv + 10, str(tv), ha="center", fontsize=8)
    axes[0].text(xi + w/2, tsv + 10, str(tsv), ha="center", fontsize=8)

# Pie — train
axes[1].pie(train_vals, labels=CLASSES, autopct="%1.1f%%", colors=CLASS_COLORS,
            startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title(f"Training Set\n(Total: {sum(train_vals):,})")

# Pie — test
axes[2].pie(test_vals, labels=CLASSES, autopct="%1.1f%%", colors=CLASS_COLORS,
            startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[2].set_title(f"Test Set\n(Total: {sum(test_vals):,})")

plt.tight_layout()
path = os.path.join(PLOT_DIR, "dataset_analysis.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")


## Cell 9 — Epoch Runner (train / eval)

In [ ]:
def run_epoch(model, loader, criterion, optimizer, training: bool):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            probs       = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / total
    acc      = 100.0 * correct / total
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), np.array(all_probs)

print("✅ run_epoch defined")


## Cell 10 — Build Model & History Store

In [ ]:
MODEL = BrainTumorNet(NUM_CLASSES).to(DEVICE)

HISTORY = {
    "train_loss": [], "val_loss": [],
    "train_acc" : [], "val_acc" : [],
    "phase2_start_epoch": EPOCHS_PHASE1 + 1,
}

best_val_acc  = 0.0
best_val_loss = float("inf")

trainable = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f"✅ Model on {DEVICE}")
print(f"   Total params     : {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"   Trainable params : {trainable:,}")


## Cell 11 — Phase 1: Frozen Backbone Training

Head + CBAM trained only. Backbone frozen. High learning rate.


In [ ]:
MODEL.freeze_backbone()
trainable_p1 = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f"[Phase 1] Backbone frozen  | Trainable params: {trainable_p1:,}\n")

optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, MODEL.parameters()),
    lr=LR_PHASE1, weight_decay=WEIGHT_DECAY,
)
scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=EPOCHS_PHASE1, eta_min=1e-6,
)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    tr_loss, tr_acc, _, _, _ = run_epoch(MODEL, TRAIN_LOADER, CRITERION, optimizer_p1, training=True)
    vl_loss, vl_acc, _, _, _ = run_epoch(MODEL, VAL_LOADER,   CRITERION, optimizer_p1, training=False)
    scheduler_p1.step()

    HISTORY["train_loss"].append(tr_loss)
    HISTORY["val_loss"  ].append(vl_loss)
    HISTORY["train_acc" ].append(tr_acc)
    HISTORY["val_acc"   ].append(vl_acc)

    lr_now = optimizer_p1.param_groups[0]["lr"]
    marker = ""
    if vl_acc > best_val_acc:
        best_val_acc  = vl_acc
        best_val_loss = vl_loss
        torch.save(MODEL.state_dict(), OUTPUT_PATH)
        marker = "  ← best saved ✅"

    print(
        f"[P1] Ep {epoch:03d}/{EPOCHS_PHASE1} | "
        f"TrainLoss={tr_loss:.4f}  TrainAcc={tr_acc:.2f}% | "
        f"ValLoss={vl_loss:.4f}  ValAcc={vl_acc:.2f}% | "
        f"LR={lr_now:.2e}{marker}"
    )

print(f"\n Phase 1 complete. Best Val Acc so far: {best_val_acc:.2f}%")


## Cell 12 — Phase 2: Full Fine-Tune

Entire model (backbone + CBAM + head) fine-tuned at a low LR with cosine warm restarts.


In [ ]:
MODEL.unfreeze_backbone()
trainable_p2 = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f"[Phase 2] Backbone unfrozen | Trainable params: {trainable_p2:,}\n")

optimizer_p2 = optim.AdamW(MODEL.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
scheduler_p2 = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p2, T_0=EPOCHS_PHASE2, T_mult=1, eta_min=1e-7,
)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    global_ep = EPOCHS_PHASE1 + epoch
    total_ep  = EPOCHS_PHASE1 + EPOCHS_PHASE2

    tr_loss, tr_acc, _, _, _ = run_epoch(MODEL, TRAIN_LOADER, CRITERION, optimizer_p2, training=True)
    vl_loss, vl_acc, _, _, _ = run_epoch(MODEL, VAL_LOADER,   CRITERION, optimizer_p2, training=False)
    scheduler_p2.step()

    HISTORY["train_loss"].append(tr_loss)
    HISTORY["val_loss"  ].append(vl_loss)
    HISTORY["train_acc" ].append(tr_acc)
    HISTORY["val_acc"   ].append(vl_acc)

    lr_now = optimizer_p2.param_groups[0]["lr"]
    marker = ""
    if vl_acc > best_val_acc or (vl_acc == best_val_acc and vl_loss < best_val_loss):
        best_val_acc  = vl_acc
        best_val_loss = vl_loss
        torch.save(MODEL.state_dict(), OUTPUT_PATH)
        marker = "  ← best saved ✅"

    print(
        f"[P2] Ep {global_ep:03d}/{total_ep} | "
        f"TrainLoss={tr_loss:.4f}  TrainAcc={tr_acc:.2f}% | "
        f"ValLoss={vl_loss:.4f}  ValAcc={vl_acc:.2f}% | "
        f"LR={lr_now:.2e}{marker}"
    )

print(f"\n Phase 2 complete. Best Val Acc: {best_val_acc:.2f}%")


## Cell 13 — Training Curves Plot

In [ ]:
epochs_x = range(1, len(HISTORY["train_loss"]) + 1)
phase_boundary = HISTORY["phase2_start_epoch"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Curves — EfficientNet-B3 + CBAM", fontsize=14, fontweight="bold")

# Loss
axes[0].plot(epochs_x, HISTORY["train_loss"], "b-o", markersize=4, linewidth=1.5, label="Train Loss")
axes[0].plot(epochs_x, HISTORY["val_loss"],   "r-s", markersize=4, linewidth=1.5, label="Val Loss")
axes[0].axvline(x=phase_boundary, color="green", linestyle="--", alpha=0.7,
                label=f"Phase 2 start (ep {phase_boundary})")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss per Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_x, HISTORY["train_acc"], "b-o", markersize=4, linewidth=1.5, label="Train Acc")
axes[1].plot(epochs_x, HISTORY["val_acc"],   "r-s", markersize=4, linewidth=1.5, label="Val Acc")
axes[1].axvline(x=phase_boundary, color="green", linestyle="--", alpha=0.7,
                label=f"Phase 2 start (ep {phase_boundary})")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy per Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim([max(0, min(HISTORY["train_acc"]) - 5), 101])

plt.tight_layout()
path = os.path.join(PLOT_DIR, "training_curves.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")


## Cell 14 — Final Evaluation on Held-Out Test Set

In [ ]:
print("Loading best weights...")
MODEL.load_state_dict(torch.load(OUTPUT_PATH, map_location=DEVICE))

_, test_acc, Y_TRUE, Y_PRED, Y_PROBS = run_epoch(
    MODEL, TEST_LOADER, CRITERION, None, training=False
)
print(f"\n Test Accuracy : {test_acc:.4f}%")
print("\n--- Per-Class Report ---")
print(classification_report(Y_TRUE, Y_PRED, target_names=CLASSES, digits=4))


## Cell 15 — Confusion Matrix

In [ ]:
cm  = confusion_matrix(Y_TRUE, Y_PRED)
cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Confusion Matrix — Test Set", fontsize=14, fontweight="bold")

for ax, data, fmt, title in zip(
    axes, [cm, cmn], ["d", ".1f"], ["Raw Counts", "Row-Normalised (%)"]
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, linecolor="gray", ax=ax,
                cbar_kws={"shrink": 0.8})
    ax.set_title(title); ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
    ax.tick_params(axis="x", rotation=30); ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
path = os.path.join(PLOT_DIR, "confusion_matrix_test.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")


## Cell 16 — Per-Class Metrics: Precision / Recall / F1 / Specificity

In [ ]:
report = classification_report(Y_TRUE, Y_PRED, target_names=CLASSES, output_dict=True)

# Specificity per class
specificity = []
for i in range(NUM_CLASSES):
    tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
    fp = cm[:, i].sum() - cm[i, i]
    specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

metric_data = {
    "Precision"   : [report[c]["precision"]  for c in CLASSES],
    "Recall"      : [report[c]["recall"]     for c in CLASSES],
    "F1-Score"    : [report[c]["f1-score"]   for c in CLASSES],
    "Specificity" : specificity,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Per-Class Metrics — Test Set", fontsize=14, fontweight="bold")
axes = axes.ravel()

for idx, (metric_name, vals) in enumerate(metric_data.items()):
    bars = axes[idx].bar(CLASSES, vals, color=CLASS_COLORS, edgecolor="black", alpha=0.85)
    axes[idx].set_ylim([max(0.70, min(vals) - 0.05), 1.015])
    axes[idx].set_ylabel(metric_name)
    axes[idx].set_title(f"Per-Class {metric_name}")
    axes[idx].tick_params(axis="x", rotation=20)
    axes[idx].grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, vals):
        axes[idx].text(bar.get_x() + bar.get_width() / 2,
                       bar.get_height() + 0.003,
                       f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
path = os.path.join(PLOT_DIR, "per_class_metrics_test.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")


## Cell 17 — AUC-ROC Curves

In [ ]:
y_bin = label_binarize(Y_TRUE, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title("AUC-ROC Curves — Test Set", fontsize=14, fontweight="bold")

auc_scores = []
for i, (cls, color) in enumerate(zip(CLASSES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], Y_PROBS[:, i])
    auc_val     = auc(fpr, tpr)
    auc_scores.append(auc_val)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f"{cls}  (AUC = {auc_val:.4f})")

macro_auc = np.mean(auc_scores)
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
ax.text(0.52, 0.10, f"Macro AUC = {macro_auc:.4f}",
        fontsize=12, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", edgecolor="gray"))

plt.tight_layout()
path = os.path.join(PLOT_DIR, "roc_curves_test.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")


## Cell 18 — All Headline Metrics Summary

In [ ]:
y_bin_full = label_binarize(Y_TRUE, classes=list(range(NUM_CLASSES)))
macro_auc  = roc_auc_score(y_bin_full, Y_PROBS, average="macro", multi_class="ovr")

METRICS = {
    "Accuracy"        : report["accuracy"],
    "Macro Precision" : report["macro avg"]["precision"],
    "Macro Recall"    : report["macro avg"]["recall"],
    "Macro F1"        : report["macro avg"]["f1-score"],
    "Weighted F1"     : report["weighted avg"]["f1-score"],
    "Macro AUC-ROC"   : macro_auc,
    "MCC"             : (matthews_corrcoef(Y_TRUE, Y_PRED) + 1) / 2,   # scaled to [0,1] for display
    "Cohen Kappa"     : cohen_kappa_score(Y_TRUE, Y_PRED),
}

names = list(METRICS.keys())
vals  = list(METRICS.values())

import matplotlib.cm as cm_mod
colors = cm_mod.RdYlGn(np.linspace(0.4, 0.9, len(names)))

fig, ax = plt.subplots(figsize=(13, 5))
bars    = ax.bar(names, vals, color=colors, edgecolor="black", alpha=0.88)
ax.set_ylim([max(0, min(vals) - 0.05), 1.03])
ax.set_ylabel("Score")
ax.set_title("Test Set — All Headline Metrics", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.004,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
path = os.path.join(PLOT_DIR, "summary_metrics.png")
plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {path}")

# Print clean table
print("\n" + "="*55)
print(f"{'Metric':<22}  {'Score':>10}  {'%':>8}")
print("="*55)
for k, v in METRICS.items():
    pct = f"{v*100:.4f}%" if k != "MCC" else f"{(v*2-1):.4f} (raw)"
    print(f"{k:<22}  {v:>10.4f}  {pct:>12}")
print("="*55)

# Save JSON
metrics_path = os.path.join(PLOT_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump({k: float(v) for k, v in METRICS.items()}, f, indent=2)
print(f"\n✅ metrics.json saved → {metrics_path}")


## Cell 19 — Benchmark Comparison Table

In [ ]:
benchmarks = [
    ("EfficientNet-B0 (baseline)",         98.14, 98.00, 98.00, 98.00),
    ("SE-EfficientNetB3 (2025)",           98.70, 98.77, 98.70, 98.70),
    ("EfficientNetB3 two-phase (2025)",    94.00, 93.60, 80.30, 86.40),  # glioma recall
    ("CEGA-EfficientNetB3 (2025) ★",       99.39, None,  None,  None),
    ("EfficientNetV2+Attention (2025)",    99.16, None,  None,  None),
    ("Ours: EfficientNetB3+CBAM",
     report["accuracy"]*100,
     report["macro avg"]["precision"]*100,
     report["macro avg"]["recall"]*100,
     report["macro avg"]["f1-score"]*100),
]

print(f"{'Model':<42} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("-"*70)
for row in benchmarks:
    name  = row[0]
    nums  = [f"{v:>7.2f}" if v is not None else "   N/A " for v in row[1:]]
    print(f"{name:<42} {'  '.join(nums)}")

print("\n★ Current SOTA on Msoud 4-class dataset")


## Cell 20 — All Plots Summary

All plots saved to `plots/` directory:

| File | Contents |
|---|---|
| `dataset_analysis.png` | Class distribution bar chart + pie charts |
| `training_curves.png` | Train/Val Loss & Accuracy per epoch, phase boundary |
| `confusion_matrix_test.png` | Raw counts + row-normalised % |
| `per_class_metrics_test.png` | Precision / Recall / F1 / Specificity per class |
| `roc_curves_test.png` | Per-class AUC-ROC + macro AUC |
| `summary_metrics.png` | All headline metrics in one bar chart |
| `metrics.json` | Machine-readable results for paper tables |


In [ ]:
import glob
plot_files = sorted(glob.glob(os.path.join(PLOT_DIR, "*.png")))
print(f"Generated {len(plot_files)} plot files:")
for f in plot_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f:<50}  {size_kb:>6.1f} KB")
